# M3GNet XAS pipeline showcase

This notebook displays the M3GNet encoder and XAS heads. It provides opt-in cells for preflight, training, evaluation, and plots. It does not start long training automatically.

## Prerequisites

From the repository root, run:

```bash
bash tutorial_omnixas/download_omnixas_raw_data.sh
export OMNIXAS_DATA_ROOT="$HOME/OmniXAS_data"
```

The script downloads and extracts FEFF data by default. VASP download is not needed for this FEFF pipeline.

The shell script requires `curl`, `md5sum`, and `tar`.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py').is_file():
    REPO_ROOT = REPO_ROOT.parent
SCRIPT = REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py'
from omnixas.model.m3gnet_xas import HEAD_HIDDEN_DIMS, FEATURE_SCALE, SPECTRUM_DIM, M3GNetXAS, XASSpectralHead

model = M3GNetXAS()
head = XASSpectralHead()
print(model)
print(f"M3GNetXAS parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"XASSpectralHead parameters: {sum(p.numel() for p in head.parameters()):,}")
print({'head_hidden_dims': HEAD_HIDDEN_DIMS, 'feature_scale': FEATURE_SCALE, 'target_dim': SPECTRUM_DIM})


## Preflight

The next cell checks FEFF rows, target dimensions, raw structure paths, and architecture. It performs no training.


In [ ]:
subprocess.run([sys.executable, str(SCRIPT), '--preflight'], cwd=REPO_ROOT, check=True)


## Training command

Run this command in a terminal after preflight. The command can take a long time. The notebook does not run it.

```bash
python tutorial_omnixas/train_m3gnet_xas_pipeline.py --preflight
python tutorial_omnixas/train_m3gnet_xas_pipeline.py --run-name m3gnet_xas_seed42 --gpu 0
```


In [ ]:
print(f"{sys.executable} {SCRIPT} --run-name m3gnet_xas_seed42 --gpu 0")


## Configure loading and evaluation

Set `RUN_DIR` to a completed run. The cell fails if a checkpoint is missing.


In [ ]:
RUN_DIR = REPO_ROOT / 'output/training/m3gnet_xas_pipeline/m3gnet_xas_seed42'
OUTPUT_ROOT = RUN_DIR.parent
RUN_NAME = RUN_DIR.name
EVALUATE = False  # Set True only for a completed named run.
if EVALUATE:
    subprocess.run([sys.executable, str(SCRIPT), '--output-root', str(OUTPUT_ROOT), '--run-name', RUN_NAME, '--evaluate'], cwd=REPO_ROOT, check=True)


## Plot validation metrics

This cell reads generated CSV files only. It does not train or select a checkpoint.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = RUN_DIR / "tuned_validation.csv"
if metrics_path.is_file():
    metrics = pd.read_csv(metrics_path)
    metrics.plot.bar(x="dataset", y="eta", legend=False, title="Tuned UniversalXAS validation eta")
    plt.tight_layout()
else:
    print(f"Set RUN_DIR to a completed run with {metrics_path.name}")
